In [2]:
import http.client
import boto3
import requests
import pandas as pd
import json
import csv
from datetime import datetime
import logging
import time
# from botocore.exceptions import NoCredentialsError

In [19]:
df = pd.read_csv("data/ucl_team_match_stats.csv")
active_team_names = df[df["match_status"] == 0]["team_name"].unique()

In [ ]:
team_stats = (
    df.groupby("team_name")
    .agg(matches_played=("match_id", "count"))
    .sort_values("matches_played", ascending=False)
    .reset_index()
)

team_stats

,team_name,matches_played
0,Paris,14
1,Real Madrid,14
2,Atleti,14
3,Bodø/Glimt,12
4,Newcastle,12
5,Liverpool,12
6,Arsenal,12
7,Galatasaray,12
8,Leverkusen,12
9,Barcelona,12


In [ ]:
team_stats = (
    df.groupby("team_name")
    .agg(
        matches_played=("match_status", lambda s: (s == 2).sum()),
        matches_remaining=("match_status", lambda s: (s == 0).sum()),
        goals_scored=("goals_scored", "sum"),
        goals_conceded=("goals_conceded", "sum"),
    )
    .assign(
        still_in_competition=lambda x: x["matches_remaining"] > 0,
        goal_difference=lambda x: x["goals_scored"] - x["goals_conceded"],
    )
    .sort_values(
        ["still_in_competition", "matches_played", "goals_scored"],
        ascending=[False, False, False]
    )
    .reset_index()
)

team_stats

,team_name,matches_played,matches_remaining,goals_scored,goals_conceded,still_in_competition,goal_difference
0,Paris,12,2,34.0,17.0,True,17.0
1,Atleti,12,2,31.0,24.0,True,7.0
2,Real Madrid,12,2,29.0,14.0,True,15.0
3,Bayern München,10,2,32.0,10.0,True,22.0
4,Barcelona,10,2,30.0,17.0,True,13.0
5,Arsenal,10,2,26.0,5.0,True,21.0
6,Liverpool,10,2,24.0,9.0,True,15.0
7,Sporting CP,10,2,22.0,14.0,True,8.0
8,Newcastle,12,0,29.0,18.0,False,11.0
9,Bodø/Glimt,12,0,22.0,22.0,False,0.0


In [ ]:
# Goals scored from played matches only
played = df[(df["team_name"].isin(active_team_names)) & (df["match_status"] == 2)]
goals = (
    played.groupby("team_name")["goals_scored"]
    .sum()
    .reset_index()
)

# Next opponent: earliest unplayed match per team
upcoming = (
    df[(df["team_name"].isin(active_team_names)) & (df["match_status"] == 0)]
    .sort_values("match_datetime")
    .groupby("team_name")
    .first()
    .reset_index()
    [["team_name", "opponent_team_name"]]
    .rename(columns={"opponent_team_name": "next_opponent"})
)

result = goals.merge(upcoming, on="team_name").sort_values("goals_scored", ascending=False).reset_index(drop=True)
result

,team_name,goals_scored,next_opponent
0,Paris,34.0,Liverpool
1,Bayern München,32.0,Real Madrid
2,Atleti,31.0,Barcelona
3,Barcelona,30.0,Atleti
4,Real Madrid,29.0,Bayern München
5,Arsenal,26.0,Sporting CP
6,Liverpool,24.0,Paris
7,Sporting CP,22.0,Arsenal


In [14]:
# Active teams: those with upcoming matches (match_status == 0)
active_team_names = df[df["match_status"] == 0]["team_name"].unique()

# Goals scored from played matches only
played = df[(df["team_name"].isin(active_team_names)) & (df["match_status"] == 2)]
goals = (
    played.groupby("team_name")["goals_conceded"]
    .sum()
    .reset_index()
)

# Next opponent: earliest unplayed match per team
upcoming = (
    df[(df["team_name"].isin(active_team_names)) & (df["match_status"] == 0)]
    .sort_values("match_datetime")
    .groupby("team_name")
    .first()
    .reset_index()
    [["team_name", "opponent_team_name"]]
    .rename(columns={"opponent_team_name": "next_opponent"})
)

result = goals.merge(upcoming, on="team_name").sort_values("goals_conceded", ascending=True).reset_index(drop=True)
result

,team_name,goals_conceded,next_opponent
0,Arsenal,5.0,Sporting CP
1,Liverpool,9.0,Paris
2,Bayern München,10.0,Real Madrid
3,Real Madrid,14.0,Bayern München
4,Sporting CP,14.0,Arsenal
5,Barcelona,17.0,Atleti
6,Paris,17.0,Liverpool
7,Atleti,24.0,Barcelona


In [ ]:
team_code = "LIV" 

team_matches = (
    df[(df["team_code"] == team_code) & (df["match_status"] == 2)]
    [["match_datetime", "opponent_team_name", "goals_scored", "goals_conceded", "is_home", "result"]]
    .sort_values("match_datetime")
    .reset_index(drop=True)
)
team_matches["venue"] = team_matches["is_home"].map({True: "Home", False: "Away"})

team_matches.drop(columns="is_home")

team_matches


,match_datetime,opponent_team_name,goals_scored,goals_conceded,is_home,result,venue
0,2025-09-17T21:00:00,Atleti,3.0,2.0,True,W,Home
1,2025-09-30T21:00:00,Galatasaray,0.0,1.0,False,L,Away
2,2025-10-22T21:00:00,Frankfurt,5.0,1.0,False,W,Away
3,2025-11-04T21:00:00,Real Madrid,1.0,0.0,True,W,Home
4,2025-11-26T21:00:00,PSV,1.0,4.0,True,L,Home
5,2025-12-09T21:00:00,Inter,1.0,0.0,False,W,Away
6,2026-01-21T21:00:00,Marseille,3.0,0.0,False,W,Away
7,2026-01-28T21:00:00,Qarabağ,6.0,0.0,True,W,Home
8,2026-03-10T18:45:00,Galatasaray,0.0,1.0,False,L,Away
9,2026-03-18T21:00:00,Galatasaray,4.0,0.0,True,W,Home


In [ ]:
team_code = "LIV"

played = df[(df["team_code"] == team_code) & (df["match_status"] == 2)]

performance = (
    played
    .assign(venue=played["is_home"].map({True: "Home", False: "Away"}))
    .groupby("venue")
    .agg(
        matches=("match_id", "count"),
        goals_scored=("goals_scored", "sum"),
        goals_conceded=("goals_conceded", "sum"),
    )
    .reset_index()
)

performance

,venue,matches,goals_scored,goals_conceded
0,Away,5,9.0,3.0
1,Home,5,15.0,6.0


In [ ]:
"""Goals conceded away for active teams"""

played_away = df[(df["team_name"].isin(active_team_names)) & (df["is_home"] == False) & (df["match_status"] == 2)]

played_away.groupby("team_name")["goals_conceded"].sum().sort_values(ascending=True).reset_index()

,team_name,goals_conceded
0,Arsenal,3.0
1,Bayern München,3.0
2,Sporting CP,3.0
3,Real Madrid,5.0
4,Liverpool,6.0
5,Barcelona,7.0
6,Atleti,8.0
7,Paris,10.0


In [ ]:
"""Goals scored at home for active teams"""

played_home = df[(df["team_name"].isin(active_team_names)) & (df["is_home"] == True) & (df["match_status"] == 2)]

played_home.groupby("team_name")["goals_scored"].sum().sort_values(ascending=False).reset_index()

,team_name,goals_scored
0,Atleti,20.0
1,Barcelona,20.0
2,Paris,18.0
3,Bayern München,16.0
4,Sporting CP,16.0
5,Liverpool,15.0
6,Real Madrid,15.0
7,Arsenal,14.0
